# Kern QLoRA - Colab Starter

Use this notebook from VS Code with Colab extension.
1) Select Kernel -> Colab -> Auto Connect
2) Run cells top to bottom.


In [ ]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))
else:
    print('device: CPU')


In [ ]:
!rm -rf /content/kern
!git clone https://github.com/OscarCode9/kern.git /content/kern
%cd /content/kern


In [ ]:
!python -m pip install -U pip
!pip install torch transformers datasets peft trl accelerate bitsandbytes sentencepiece


In [ ]:
!python prepare_finetune_dataset_csn.py --target-kept 20000 --valid-ratio 0.05 --out-dir data/finetune_csn20k_v2 --overwrite


In [ ]:
# T4-safe first run: start with 5k rows to validate stability.
!python train_qwen_qlora_t4.py \
  --model-name Qwen/Qwen3.5-9B \
  --train-file data/finetune_csn20k_v2/train_qwen_chat.jsonl \
  --valid-file data/finetune_csn20k_v2/valid_qwen_chat.jsonl \
  --output-dir outputs/qwen-kern-qlora-t4 \
  --max-train-samples 5000 \
  --max-valid-samples 500 \
  --num-train-epochs 1 \
  --per-device-batch-size 1 \
  --gradient-accumulation-steps 16 \
  --max-seq-length 512 \
  --save-steps 100 \
  --eval-steps 100 \
  --resume-from-checkpoint auto
